In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [2]:
df = pd.read_csv("../data/processed.csv")

In [3]:
df.head()

,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustSpeed,RainTomorrow,Pressure_mean,Pressure_diff,WindSpeed_mean,...,Humidity_diff,Cloud_mean,Cloud_diff,Temp_range,Temp_diff,Month,WindDir9am_angle,WindDir3pm_angle,WindGustDir_angle,ClimateZone
0,10.8,21.2,0.0,1.8,6.60,22.0,0,1026.55,-3.7,8.0,...,-18.0,4.48,0.06,18.35,5.3,4,67.5,315.0,67.5,4
1,3.7,19.0,0.0,1.4,7.61,24.0,0,1022.65,-3.1,5.5,...,-43.0,4.48,0.06,14.05,8.9,7,0.0,22.5,0.0,4
2,9.6,15.8,0.0,2.6,7.61,52.0,1,1014.70,-6.4,14.5,...,16.0,4.48,0.06,15.05,0.7,7,22.5,45.0,45.0,4
3,10.1,15.5,16.6,0.8,7.61,50.0,1,1007.70,0.6,21.5,...,-16.0,4.48,0.06,12.60,2.8,7,315.0,315.0,22.5,4
4,11.2,16.2,1.8,0.6,7.61,30.0,1,1018.80,0.8,14.0,...,-19.0,4.48,0.06,14.15,2.3,7,292.5,292.5,315.0,4


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145230 entries, 0 to 145229
Data columns (total 22 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   MinTemp            145230 non-null  float64
 1   MaxTemp            145230 non-null  float64
 2   Rainfall           145230 non-null  float64
 3   Evaporation        145230 non-null  float64
 4   Sunshine           145230 non-null  float64
 5   WindGustSpeed      145230 non-null  float64
 6   RainTomorrow       145230 non-null  int64  
 7   Pressure_mean      145230 non-null  float64
 8   Pressure_diff      145230 non-null  float64
 9   WindSpeed_mean     145230 non-null  float64
 10  WindSpeed_diff     145230 non-null  float64
 11  Humidity_mean      145230 non-null  float64
 12  Humidity_diff      145230 non-null  float64
 13  Cloud_mean         145230 non-null  float64
 14  Cloud_diff         145230 non-null  float64
 15  Temp_range         145230 non-null  float64
 16  Te

In [5]:
df.shape

(145230, 22)

In [6]:
# train test split
X = df.drop('RainTomorrow', axis = 1)
y = df['RainTomorrow']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

### Traning a logistic regression

In [17]:
model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train, y_train)

C:\Users\pradip\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(class_weight='balanced', max_iter=1000)

In [9]:
y_pred = model.predict(X_test)

In [10]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred))

[[17750  4941]
 [ 1543  4812]]
              precision    recall  f1-score   support

           0       0.92      0.78      0.85     22691
           1       0.49      0.76      0.60      6355

    accuracy                           0.78     29046
   macro avg       0.71      0.77      0.72     29046
weighted avg       0.83      0.78      0.79     29046

ROC-AUC: 0.7697237622086708


---
### Traning Randome forest classifier

In [11]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)

rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200,
                       random_state=42)

In [12]:
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:,1]

In [13]:
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

[[21882   809]
 [ 3412  2943]]
              precision    recall  f1-score   support

           0       0.87      0.96      0.91     22691
           1       0.78      0.46      0.58      6355

    accuracy                           0.85     29046
   macro avg       0.82      0.71      0.75     29046
weighted avg       0.85      0.85      0.84     29046

ROC-AUC: 0.8914806180152116


---
### Traning XGBClassifier

In [14]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight= (len(y_train[y_train==0]) / len(y_train[y_train==1])),
    random_state=42
)

xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, ...)

In [15]:
y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:,1]

In [16]:
print(confusion_matrix(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))

[[18551  4140]
 [ 1337  5018]]
              precision    recall  f1-score   support

           0       0.93      0.82      0.87     22691
           1       0.55      0.79      0.65      6355

    accuracy                           0.81     29046
   macro avg       0.74      0.80      0.76     29046
weighted avg       0.85      0.81      0.82     29046

ROC-AUC: 0.8908201281534864


---
## With index
###  LogisticRegression
                precision    recall  f1-score   support

           0       0.92      0.78      0.84     22691
           1       0.49      0.75      0.59      6355

    accuracy                           0.77     29046
   macro avg       0.70      0.76      0.72     29046
weighted avg       0.82      0.77      0.79     29046

ROC-AUC: 0.7632184847425618

### RandomForestClassifier
                precision    recall  f1-score   support

           0       0.87      0.96      0.91     22691
           1       0.79      0.46      0.58      6355

    accuracy                           0.86     29046
   macro avg       0.83      0.71      0.75     29046
weighted avg       0.85      0.86      0.84     29046

ROC-AUC: 0.8921935311195691

###  XGBClassifier
                precision    recall  f1-score   support

           0       0.93      0.82      0.87     22691
           1       0.55      0.79      0.65      6355

    accuracy                           0.81     29046
   macro avg       0.74      0.80      0.76     29046
weighted avg       0.85      0.81      0.82     29046

ROC-AUC: 0.8924310151007302

## Without index
###  LogisticRegression
                precision    recall  f1-score   support

           0       0.92      0.78      0.85     22691
           1       0.49      0.76      0.60      6355

    accuracy                           0.78     29046
   macro avg       0.71      0.77      0.72     29046
weighted avg       0.83      0.78      0.79     29046

ROC-AUC: 0.7697237622086708

### RandomForestClassifier
                precision    recall  f1-score   support

           0       0.87      0.96      0.91     22691
           1       0.78      0.46      0.58      6355

    accuracy                           0.85     29046
   macro avg       0.82      0.71      0.75     29046
weighted avg       0.85      0.85      0.84     29046

ROC-AUC: 0.8914806180152116

###  XGBClassifier
                precision    recall  f1-score   support

           0       0.93      0.82      0.87     22691
           1       0.55      0.79      0.65      6355

    accuracy                           0.81     29046
   macro avg       0.74      0.80      0.76     29046
weighted avg       0.85      0.81      0.82     29046

ROC-AUC: 0.8908201281534864